# RepoExec generation on an FMLe GPU node

This notebook performs generation only. It runs the first 30 `full_context` tasks for all configured models and the `raw`, `ast`, and `reduced_ast` representations. Docker evaluation remains on the local Windows machine.

Run the cells in order. The long generation cell keeps running in the Jupyter kernel if the browser disconnects. Do not restart or stop the kernel while it is running.

## 1. Configuration

In [ ]:
import json
import os
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10+ is required; this kernel uses {sys.version}")

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "repoexec_baseline").is_dir() and (path / "requirements-baseline.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the cloned llm-experiment repository.")

STORAGE_ROOT = Path(os.environ.get("FMLE_STORAGE_ROOT", PROJECT_ROOT)).expanduser().resolve()
RUNS_ROOT = STORAGE_ROOT / "repoexec-runs"
OLLAMA_RUNTIME_DIR = STORAGE_ROOT / "ollama-runtime"
OLLAMA_MODELS_DIR = STORAGE_ROOT / "ollama-models"
HF_HOME = STORAGE_ROOT / "hf-cache"
RUN_PREFIX = "fmle-generation30"
TASK_LIMIT = 30
NUM_RETURN_SEQUENCES = 5
PULL_MODELS = True

MODELS = [
    "qwen2.5-coder:1.5b-base",
    "qwen2.5-coder:3b-base",
    "qwen2.5-coder:7b-base",
    "deepseek-coder:1.3b-base-q4_K_M",
    "deepseek-coder:6.7b-base-q4_K_M",
    "deepseek-coder-v2:16b-lite-base-q4_K_M",
    "codegemma:2b-code-q4_K_M",
    "codegemma:7b-code-q4_K_M",
]
REPRESENTATIONS = ["raw", "ast", "reduced_ast"]

for directory in (RUNS_ROOT, OLLAMA_RUNTIME_DIR, OLLAMA_MODELS_DIR, HF_HOME):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["HF_DATASETS_OFFLINE"] = "0"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
os.environ["PATH"] = f"{OLLAMA_RUNTIME_DIR / 'bin'}:{os.environ['PATH']}"
os.environ["LD_LIBRARY_PATH"] = f"{OLLAMA_RUNTIME_DIR / 'lib' / 'ollama'}:{os.environ.get('LD_LIBRARY_PATH', '')}"

TIMINGS = {}
print(f"Host: {socket.gethostname()}")
print(f"Python: {sys.executable} ({sys.version.split()[0]})")
print(f"Project: {PROJECT_ROOT}")
print(f"Storage: {STORAGE_ROOT}")
print(f"Runs: {RUNS_ROOT}")

## 2. Verify the dedicated GPU

This cell must list at least one NVIDIA GPU. Do not continue if it fails or if you are still on `login01` without an allocated GPU.

In [ ]:
gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError(f"No NVIDIA GPU is visible:\n{gpu_check.stderr}")
print(gpu_check.stdout)
subprocess.run(["nvidia-smi"], check=True)

## 3. Install Python dependencies

Packages are installed into the active Jupyter kernel environment. No `sudo` is used.

In [ ]:
started = time.perf_counter()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-baseline.txt")],
    check=True,
)
TIMINGS["python_dependencies_seconds"] = time.perf_counter() - started
print(f"Python dependencies: {TIMINGS['python_dependencies_seconds']:.2f}s")

## 4. Install Ollama without sudo

The official Linux bundle is downloaded and extracted under the selected storage root. Re-running this cell uses the existing installation.

In [ ]:
started = time.perf_counter()
ollama_binary = shutil.which("ollama")
if ollama_binary is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "repoexec_baseline.install_ollama_user",
            "--install-dir",
            str(OLLAMA_RUNTIME_DIR),
        ],
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        check=True,
    )
    ollama_binary = shutil.which("ollama")
if ollama_binary is None:
    raise RuntimeError("Ollama installation finished but its binary is not in PATH.")
TIMINGS["ollama_install_seconds"] = time.perf_counter() - started
print(f"Ollama: {ollama_binary}")
subprocess.run([ollama_binary, "--version"], env=os.environ.copy(), check=True)
print(f"Ollama installation check: {TIMINGS['ollama_install_seconds']:.2f}s")

## 5. Start the local Ollama service

In [ ]:
import requests

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_LOG = RUNS_ROOT / f"{RUN_PREFIX}-ollama.log"
OLLAMA_PROCESS = None
OLLAMA_LOG_HANDLE = None

def ollama_ready():
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        response.raise_for_status()
        return True
    except requests.RequestException:
        return False

started = time.perf_counter()
if not ollama_ready():
    OLLAMA_LOG_HANDLE = OLLAMA_LOG.open("a", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        [ollama_binary, "serve"],
        stdout=OLLAMA_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        start_new_session=True,
    )
    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(1)

if not ollama_ready():
    log_tail = OLLAMA_LOG.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Ollama did not start. Log tail:\n{log_tail}")
TIMINGS["ollama_start_seconds"] = time.perf_counter() - started
print(f"Ollama API is ready at {OLLAMA_BASE_URL} ({TIMINGS['ollama_start_seconds']:.2f}s)")

## 6. Download or verify models

This is sequential and has no retry logic. Existing models are not downloaded again.

In [ ]:
started = time.perf_counter()
for model in MODELS:
    available = subprocess.run(
        [ollama_binary, "show", model],
        env=os.environ.copy(),
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0
    if available:
        print(f"Already available: {model}")
    elif PULL_MODELS:
        print(f"Pulling: {model}", flush=True)
        subprocess.run([ollama_binary, "pull", model], env=os.environ.copy(), check=True)
    else:
        raise RuntimeError(f"Missing Ollama model: {model}")
TIMINGS["model_pull_seconds"] = time.perf_counter() - started
print(f"Model preparation: {TIMINGS['model_pull_seconds'] / 60:.2f} min")

## 7. Run all generations

This launches 8 models x 3 representations x 30 tasks x 5 candidates = 3,600 Ollama requests. Completed model/representation runs are skipped when the cell is run again. An incomplete run is regenerated from its beginning. Generation requests are never retried automatically.

In [ ]:
generation_command = [
    sys.executable,
    "-m",
    "repoexec_baseline.run_generation_matrix",
    "--models",
    *MODELS,
    "--representations",
    *REPRESENTATIONS,
    "--subset",
    "full_context",
    "--task-limit",
    str(TASK_LIMIT),
    "--runs-root",
    str(RUNS_ROOT),
    "--run-prefix",
    RUN_PREFIX,
    "--num-return-sequences",
    str(NUM_RETURN_SEQUENCES),
    "--max-new-tokens",
    "256",
    "--do-sample",
    "--temperature",
    "0.2",
    "--top-p",
    "0.95",
    "--seed",
    "42",
    "--ollama-base-url",
    OLLAMA_BASE_URL,
    "--ollama-keep-alive",
    "30m",
]
generation_env = os.environ.copy()
generation_env["PYTHONUNBUFFERED"] = "1"
started = time.perf_counter()
subprocess.run(generation_command, cwd=PROJECT_ROOT, env=generation_env, check=True)
TIMINGS["generation_matrix_seconds"] = time.perf_counter() - started
print(f"Generation matrix: {TIMINGS['generation_matrix_seconds'] / 3600:.2f} h")

## 8. Inspect status, timing, and transfer bundle

In [ ]:
from IPython.display import FileLink, display

summary_path = RUNS_ROOT / f"{RUN_PREFIX}-generation-summary.json"
bundle_path = RUNS_ROOT / f"{RUN_PREFIX}-generation-bundle.tar.gz"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

measured_total = sum(float(value) for value in TIMINGS.values())
notebook_timing = {
    **TIMINGS,
    "total_measured_work_seconds": measured_total,
    "total_measured_work_hours": measured_total / 3600,
}
timing_path = RUNS_ROOT / f"{RUN_PREFIX}-notebook-timing.json"
timing_path.write_text(json.dumps(notebook_timing, indent=2), encoding="utf-8")
summary["notebook_active_timings"] = notebook_timing
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
from repoexec_baseline.run_generation_matrix import create_bundle
create_bundle(bundle_path, summary_path, RUNS_ROOT, summary["runs"])

print(f"Status: {summary['status']}")
print(f"Completed entries: {sum(row['status'] in ('completed', 'skipped_existing') for row in summary['runs'])}/{len(MODELS) * len(REPRESENTATIONS)}")
print(f"Measured active work: {measured_total / 3600:.2f} h")
print(f"Summary: {summary_path}")
print(f"Timing: {timing_path}")
print(f"Bundle: {bundle_path}")
display(FileLink(str(bundle_path)))

After downloading and extracting the bundle into the local `runs` directory, evaluate it on Windows with:

```powershell
.\.venv\Scripts\python.exe -m repoexec_baseline.evaluate_generation_matrix `
  --matrix-summary runs\fmle-generation30-generation-summary.json `
  --repoexec-dir RepoExec
```

## 9. Optional cleanup

Run this only after generation is complete if you want to stop the Ollama process without stopping the Jupyter instance.

In [ ]:
if OLLAMA_PROCESS is not None and OLLAMA_PROCESS.poll() is None:
    OLLAMA_PROCESS.terminate()
    OLLAMA_PROCESS.wait(timeout=30)
if OLLAMA_LOG_HANDLE is not None and not OLLAMA_LOG_HANDLE.closed:
    OLLAMA_LOG_HANDLE.close()
print("Notebook-owned Ollama process stopped.")